# Validate Routing Outputs

Run lightweight checks on generated routing features, sparse SLX edge lists, and yearly POI files.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

def find_project_dir(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "README.md").exists() and (path / "ANAL").exists() and (path / "TOOLS").exists():
            return path
    raise FileNotFoundError("Could not find project root")


PROJECT_DIR = find_project_dir(Path.cwd())
ANAL_DATA = PROJECT_DIR / "ANAL" / "data"
ROUTING_DATA = ANAL_DATA / "routing"
OSM_DIR = PROJECT_DIR / "TOOLS" / "osm-data"
YEARS = range(2015, 2026)

ACTIVE_CELLS_PATH = ROUTING_DATA / "inputs" / "active_routing_cells_100m.parquet"
POPULATION_BACKCAST_PATH = ANAL_DATA / "population_backcast_100m_quarterly.parquet"
REPORT_ROOT = ROUTING_DATA / "reports"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
active_cells = pd.read_parquet(ACTIVE_CELLS_PATH)
active_grid_ids = set(active_cells["grid_id"])


def check(condition: bool, message: str) -> dict:
    return {"check": message, "status": "OK" if condition else "CHECK"}


def load_yearly_population_backcast(year: int) -> pd.Series:
    population = pd.read_parquet(
        POPULATION_BACKCAST_PATH,
        columns=["grid_id", "year", "population_backcast"],
        filters=[("year", "==", year)],
    )
    population = population.drop_duplicates(["grid_id", "year"])[["grid_id", "population_backcast"]].copy()
    population["population_backcast"] = pd.to_numeric(population["population_backcast"], errors="coerce").fillna(0.0).astype(float)
    return population.set_index("grid_id")["population_backcast"].rename("population_backcast")


def validate_pois(year: int) -> list[dict]:
    path = OSM_DIR / f"austria-{year}-pois.geoparquet"
    if not path.exists():
        return [check(False, f"{path.name} exists")]
    pois = gpd.read_parquet(path)
    expected_types = {"motorway_exit", "rail_station", "regional_centre", "urban_centre", "higher_education", "pt_stop"}
    pt_stops = pois[pois["poi_type"] == "pt_stop"].copy() if "poi_type" in pois.columns else gpd.GeoDataFrame()
    results = [
        check(len(pois) > 0, "POI file has rows"),
        check("poi_type" in pois.columns, "POI file has poi_type"),
        check(set(pois["poi_type"]).issubset(expected_types), "POI types are expected"),
        check(expected_types.issubset(set(pois["poi_type"])), "all routed POI types are present"),
        check("static_destination" in pois.columns, "POI file has static_destination flag"),
        check("pt_departures_weekday" in pois.columns, "POI file has pt_departures_weekday"),
        check(pois.geometry.notna().all(), "POI geometries are present"),
    ]
    if "pt_departures_weekday" in pois.columns and not pt_stops.empty:
        pt_departures = pd.to_numeric(pt_stops["pt_departures_weekday"], errors="coerce")
        results.extend([
            check(pt_departures.notna().all(), "PT stops have weekday departures"),
            check((pt_departures >= 0).all(), "PT stop weekday departures are non-negative"),
        ])
    return results


def validate_nearest_features(path: Path) -> list[dict]:
    if not path.exists():
        return [check(False, f"{path.name} exists")]
    nearest = pd.read_parquet(path)
    results = [
        check(len(nearest) == len(active_cells), "nearest feature row count matches active cells"),
        check("has_pt_stop_5min_walk" in nearest.columns, "nearest output has has_pt_stop_5min_walk"),
        check("pt_departures_5min_walk" in nearest.columns, "nearest output has pt_departures_5min_walk"),
    ]
    if "has_pt_stop_5min_walk" in nearest.columns:
        results.append(check(nearest["has_pt_stop_5min_walk"].notna().all(), "has_pt_stop_5min_walk has no nulls"))
    if "pt_departures_5min_walk" in nearest.columns:
        departures = pd.to_numeric(nearest["pt_departures_5min_walk"], errors="coerce")
        results.extend([
            check(departures.notna().all(), "pt_departures_5min_walk is numeric"),
            check((departures >= 0).all(), "pt_departures_5min_walk is non-negative"),
        ])
    required_columns = {"has_pt_stop_5min_walk", "pt_departures_5min_walk"}
    if required_columns.issubset(set(nearest.columns)):
        has_pt = nearest["has_pt_stop_5min_walk"].astype(bool)
        departures = pd.to_numeric(nearest["pt_departures_5min_walk"], errors="coerce").fillna(0.0)
        results.extend([
            check((departures[~has_pt] == 0.0).all(), "cells without PT stop have zero departures"),
            check((departures[has_pt] > 0).all(), "cells with PT stop have positive departures"),
        ])
    return results


def validate_population_accessibility(path: Path, year: int) -> list[dict]:
    if not path.exists():
        return [check(False, f"{path.name} exists")]
    potentials = pd.read_parquet(path)
    required_columns = {"grid_id", "pop_access_15min", "pop_access_30min", "year", "created_at"}
    results = [
        check(len(potentials) == len(active_cells), "population accessibility row count matches active cells"),
        check(required_columns.issubset(set(potentials.columns)), "population accessibility has required columns"),
    ]
    if "created_at" in potentials.columns:
        results.append(check(potentials["created_at"].notna().all(), "population accessibility has created_at values"))
    if "year" in potentials.columns:
        results.append(check((pd.to_numeric(potentials["year"], errors="coerce") == year).all(), "population accessibility year column matches file year"))
    if {"pop_access_15min", "pop_access_30min"}.issubset(set(potentials.columns)):
        pop_access_15 = pd.to_numeric(potentials["pop_access_15min"], errors="coerce")
        pop_access_30 = pd.to_numeric(potentials["pop_access_30min"], errors="coerce")
        results.extend([
            check(pop_access_15.notna().all(), "pop_access_15min is numeric"),
            check(pop_access_30.notna().all(), "pop_access_30min is numeric"),
            check((pop_access_15 >= 0).all(), "pop_access_15min is non-negative"),
            check((pop_access_30 >= 0).all(), "pop_access_30min is non-negative"),
            check((pop_access_30 >= pop_access_15).all(), "pop_access_30min is at least pop_access_15min"),
        ])
        own_population = active_cells[["grid_id"]].copy()
        own_population["own_population_backcast"] = own_population["grid_id"].map(load_yearly_population_backcast(year)).fillna(0.0).astype(float)
        merged = potentials[["grid_id", "pop_access_15min", "pop_access_30min"]].merge(own_population, on="grid_id", how="left")
        pop_access_15 = pd.to_numeric(merged["pop_access_15min"], errors="coerce").fillna(0.0)
        pop_access_30 = pd.to_numeric(merged["pop_access_30min"], errors="coerce").fillna(0.0)
        own_population_values = pd.to_numeric(merged["own_population_backcast"], errors="coerce").fillna(0.0)
        results.extend([
            check((pop_access_15 >= own_population_values).all(), "pop_access_15min includes at least own population"),
            check((pop_access_30 >= own_population_values).all(), "pop_access_30min includes at least own population"),
        ])
    return results


def validate_slx_edges(path: Path) -> list[dict]:
    if not path.exists():
        return [check(False, f"{path.name} exists")]
    edges = pd.read_parquet(path)
    row_sums = edges.groupby("origin_grid_id")["weight_rowstd"].sum()
    return [
        check((edges["origin_grid_id"] != edges["destination_grid_id"]).all(), "no self-neighbors"),
        check((edges["network_distance_m"] > 0).all(), "network distances are positive"),
        check((edges["network_distance_m"] <= 1000).all(), "main SLX cutoff is respected"),
        check((edges["weight_raw"] > 0).all(), "raw weights are positive"),
        check(np.allclose(row_sums, 1.0, atol=1e-6), "row-standardized weights sum to 1"),
        check(set(edges["origin_grid_id"]).issubset(active_grid_ids), "origin IDs exist in active cells"),
        check(set(edges["destination_grid_id"]).issubset(active_grid_ids), "destination IDs exist in active cells"),
    ]


In [ ]:
records = []
for year in YEARS:
    for result in validate_pois(year):
        records.append({"year": year, "product": "pois", **result})
    nearest_path = ROUTING_DATA / "features" / str(year) / "nearest_infrastructure_100m.parquet"
    if nearest_path.exists():
        for result in validate_nearest_features(nearest_path):
            records.append({"year": year, "product": "nearest_features", **result})
    potentials_path = ROUTING_DATA / "features" / str(year) / "accessibility_potentials_100m.parquet"
    if potentials_path.exists():
        for result in validate_population_accessibility(potentials_path, year):
            records.append({"year": year, "product": "population_accessibility", **result})
    slx_path = ROUTING_DATA / "matrices" / str(year) / "W_local_drive_1km_hl500m_edges.parquet"
    if slx_path.exists():
        for result in validate_slx_edges(slx_path):
            records.append({"year": year, "product": "slx_edges", **result})

validation = pd.DataFrame(records)
validation.to_csv(REPORT_ROOT / "routing_validation_summary.csv", index=False)
validation
